In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

from transformers_blocks import Block

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# ================== DATA PREPARATION ==================
corpus = [
    "hello friends how are you",
    "the tea is very hot",
    "my name is Adi",
    "the roads of Bangalore are busy always",
    "it is raining in Indranagar",
    "the train is late again",
    "i like having cars",
    "onam is my favorite festival",
    "diwali brings light and sweets",
    "india won cricket match",
    # Add more sentences to improve quality
    "bangalore has many it companies",
    "i love eating dosa and idli",
    "the weather is pleasant today",
    "cricket is very popular in india",
    "festival season brings joy and lights",
]

# Better tokenization
corpus = [s.strip() + " <END>" for s in corpus]
text = "\n".join(corpus)
print("Total text length (characters):", len(text))
print("Sample text:\n", text[:500])

# Word-level vocabulary
words = sorted(list(set(text.split())))
vocab_size = len(words)
print(f"Vocabulary size: {vocab_size}")

word2idx = {w: i for i, w in enumerate(words)}
idx2word = {i: w for w, i in word2idx.items()}

# Convert text to tensor
data = torch.tensor([word2idx[w] for w in text.split()], dtype=torch.long)
print(f"Data tensor shape: {data.shape}")

# ================== HYPERPARAMETERS ==================
block_size = 8          # context length
embedding_dim = 64      # increased from 32
n_heads = 4
n_layers = 4            # increased from 2
dropout = 0.1
batch_size = 16
lr = 3e-4
epochs = 5000           # increased significantly
eval_interval = 500
eval_iters = 20

# ================== TRAIN/VAL SPLIT ==================
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split='train'):
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    return x, y

@torch.no_grad()
def estimate_loss(model):
    model.eval()
    losses = {}
    for split in ['train', 'val']:
        loss_sum = 0.0
        for _ in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            loss_sum += loss.item()
        losses[split] = loss_sum / eval_iters
    model.train()
    return losses

# ================== MODEL ==================
class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(block_size, embedding_dim)
        
        self.blocks = nn.Sequential(*[
            Block(embedding_dim, block_size, n_heads, dropout) 
            for _ in range(n_layers)
        ])
        
        self.ln_f = nn.LayerNorm(embedding_dim)
        self.head = nn.Linear(embedding_dim, vocab_size)
        
        # Weight tying (optional but good practice)
        # self.token_embedding.weight = self.head.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, idx, max_new_tokens=50, temperature=0.8, top_k=10):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] if idx.size(1) > block_size else idx
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx


# ================== TRAINING ==================
model = TinyGPT()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

print(f"Model has {sum(p.numel() for p in model.parameters())} parameters")

for step in range(epochs):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if step % eval_interval == 0 or step == epochs - 1:
        losses = estimate_loss(model)
        print(f"Step {step:4d} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")

# ================== GENERATION ==================
print("\n" + "="*60)
print("GENERATED TEXT:")
print("="*60)

# Try different starting words
start_words = ["hello", "raining", "bangalore", "india", "diwali", "onam"]

for start in start_words:
    if start in word2idx:
        context = torch.tensor([[word2idx[start]]], dtype=torch.long)
        out = model.generate(context, max_new_tokens=40, temperature=0.85, top_k=15)
        
        generated = " ".join([idx2word[int(i)] for i in out[0]])
        print(f"\nStart: '{start}' →")
        print(generated)
        print("-" * 50)

# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'word2idx': word2idx,
    'idx2word': idx2word,
    'vocab_size': vocab_size,
}, 'tinygpt_model.pth')

print("\nModel saved as 'tinygpt_model.pth'")